# Trustpilot Ingestion Notebook (Data Solutions API)

In [0]:
# Databricks notebook source
# Trustpilot Public Reviews Ingestion Notebook

import hashlib
import time
from datetime import datetime, timezone
from typing import Dict, Iterator, List, Optional, Tuple
from urllib.parse import urlparse

import requests
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
CONFIG = {
    # Public Trustpilot API base
    "api_base": "https://api.trustpilot.com/v1",

    # Databricks secret setup
    "secret_scope": "trustpilot",
    "secret_key": "api_key",

    # Pagination / retry setup
    "per_page": 100,
    "max_retries": 5,
    "rate_limit_sleep_seconds": 60,
    "backoff_seconds": 2,

    # Output setup
    "catalog": "avant_users",
    "schema": "kaley_ubellacker",
    "bronze_table": "trustpilot_reviews_bronze",
    "csv_output": "/Volumes/avant_users/kaley_ubellacker/sentiment_analysis/trustpilot_master_reviews",
    
    # all reviews replacement during refresh
    "write_mode": "overwrite",
}


COMPANIES = [
    {"company": "avant", "domain": "avant.com"},
    {"company": "mission_lane", "domain": "missionlane.com"},
    {"company": "merrick_bank", "domain": "merrickbank.com"},
    {"company": "onemain_financial", "domain": "onemainfinancial.com"},
    {"company": "concora", "domain": "concoracredit.com"},
    {"company": "indigo", "domain": "indigocard.com"},
    {"company": "credit_one", "domain": "creditonebank.com"},
]

In [0]:
def get_api_key() -> str:
    key = dbutils.secrets.get(
        scope=CONFIG["secret_scope"],
        key=CONFIG["secret_key"],
    )

    key = key.strip()

    # Clean common accidental copy/paste prefixes.
    if key.lower().startswith("apikey:"):
        key = key.split(":", 1)[1].strip()

    if key.lower().startswith("apikey="):
        key = key.split("=", 1)[1].strip()

    return key.strip().strip('"').strip("'")


def normalize_domain(value: Optional[str]) -> str:
    if not value:
        return ""

    value = str(value).strip().lower()

    if "://" in value:
        parsed = urlparse(value)
        value = parsed.netloc or parsed.path

    value = value.split("/")[0]
    value = value.split("?")[0]
    value = value.strip(".")
    value = value.removeprefix("www.")

    return value

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONFIG['catalog']}.{CONFIG['schema']}")
api_key = get_api_key()
all_rows: List[Dict] = []

for item in COMPANIES:
    print(f"Fetching: {item['company']} ({item['domain']})")
    all_rows.extend(fetch_company_reviews(item["company"], item["domain"], api_key))

schema = T.StructType([
    T.StructField("review_id", T.StringType()),
    T.StructField("company", T.StringType()),
    T.StructField("domain", T.StringType()),
    T.StructField("business_unit_id", T.StringType()),
    T.StructField("rating", T.IntegerType()),
    T.StructField("title", T.StringType()),
    T.StructField("text", T.StringType()),
    T.StructField("language", T.StringType()),
    T.StructField("created_at", T.StringType()),
    T.StructField("updated_at", T.StringType()),
    T.StructField("consumer_country", T.StringType()),
    T.StructField("consumer_hash", T.StringType()),
    T.StructField("source", T.StringType()),
    T.StructField("ingested_at", T.StringType()),
])

raw_df = spark.createDataFrame(all_rows, schema=schema)
clean_df = (
    raw_df.withColumn("created_ts", F.to_timestamp("created_at"))
    .withColumn("updated_ts", F.to_timestamp("updated_at"))
    .dropDuplicates(["review_id", "company"])
)

clean_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(
    f"{CONFIG['catalog']}.{CONFIG['schema']}.{CONFIG['bronze_table']}"
)
clean_df.coalesce(1).write.mode("overwrite").option("header", True).csv(CONFIG["csv_output"])

spark.sql(f"OPTIMIZE {CONFIG['catalog']}.{CONFIG['schema']}.{CONFIG['bronze_table']} ZORDER BY (company, created_ts)")
print(f"Ingested {clean_df.count()} reviews")

In [ ]:
def request_with_retry(url: str, headers: Dict[str, str], params: Dict) -> Dict:
    last_response = None

    for attempt in range(1, CONFIG["max_retries"] + 1):
        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=60,
        )

        last_response = response

        if response.status_code == 200:
            return response.json() if response.text else {}

        if response.status_code == 429:
            print(
                f"Rate limited. Sleeping {CONFIG['rate_limit_sleep_seconds']} seconds. "
                f"url={response.url}"
            )
            time.sleep(CONFIG["rate_limit_sleep_seconds"])
            continue

        if response.status_code in (500, 502, 503, 504):
            sleep_for = CONFIG["backoff_seconds"] * (2 ** (attempt - 1))
            print(
                f"Server error {response.status_code}. "
                f"Attempt {attempt}/{CONFIG['max_retries']}. "
                f"Sleeping {sleep_for} seconds."
            )
            time.sleep(sleep_for)
            continue

        raise RuntimeError(
            "Trustpilot API request failed. "
            f"status={response.status_code}; "
            f"url={response.url}; "
            f"body={response.text[:1000]}"
        )

    raise RuntimeError(
        "Trustpilot API request failed after retries. "
        f"status={last_response.status_code if last_response else None}; "
        f"url={last_response.url if last_response else url}; "
        f"body={(last_response.text if last_response else '')[:1000]}"
    )

In [ ]:
def find_business_unit_id(domain: str, api_key: str) -> str:
    """
    Step 1:
    Use issuer domain to find the Trustpilot business unit ID.

    Public endpoint:
      GET /business-units/find?name={domain}
      Header: apikey
    """
    clean_domain = normalize_domain(domain)

    url = f"{CONFIG['api_base']}/business-units/find"
    headers = {
        "apikey": api_key,
        "Accept": "application/json",
    }
    params = {
        "name": clean_domain,
    }

    payload = request_with_retry(
        url=url,
        headers=headers,
        params=params,
    )

    business_unit_id = payload.get("id")

    if not business_unit_id:
        raise ValueError(
            f"Trustpilot business-units/find did not return an id for domain={clean_domain}. "
            f"payload={payload}"
        )

    print(
        f"Found business unit for {clean_domain}: "
        f"{business_unit_id}; "
        f"displayName={payload.get('displayName')}; "
        f"identifying={(payload.get('name') or {}).get('identifying')}"
    )

    return business_unit_id

In [ ]:
def iter_public_reviews(
    business_unit_id: str,
    api_key: str,
    per_page: int = 100,
) -> Iterator[Dict]:
    """
    Step 2:
    Use business unit ID to fetch public reviews.

    Public endpoint:
      GET /business-units/{businessUnitId}/reviews
      Header: apikey

    Pagination:
      page
      perPage
    """
    url = f"{CONFIG['api_base']}/business-units/{business_unit_id}/reviews"
    headers = {
        "apikey": api_key,
        "Accept": "application/json",
    }

    page = 1

    while True:
        params = {
            "page": page,
            "perPage": per_page,
        }

        payload = request_with_retry(
            url=url,
            headers=headers,
            params=params,
        )

        reviews = payload.get("reviews", []) or []

        if not reviews:
            break

        print(
            f"businessUnitId={business_unit_id}: "
            f"page={page}; fetched={len(reviews)}"
        )

        for review in reviews:
            yield review

        if len(reviews) < per_page:
            break

        page += 1

In [ ]:
def flatten_review(
    review: Dict,
    company: str,
    domain: str,
    business_unit_id: str,
) -> Dict:
    """
    Preserve the original downstream table contract.

    Original columns:
      review_id
      company
      domain
      business_unit_id
      rating
      title
      text
      language
      created_at
      updated_at
      consumer_country
      consumer_hash
      source
      ingested_at
    """
    consumer = review.get("consumer") or {}

    display_name = consumer.get("displayName", "")
    consumer_hash = (
        hashlib.sha256(display_name.encode("utf-8")).hexdigest()
        if display_name
        else None
    )

    # Public reviews may not always include countryCode.
    # Keep the original column name, but fall back to displayLocation if countryCode is unavailable.
    consumer_country = (
        consumer.get("countryCode")
        or consumer.get("displayLocation")
    )

    return {
        "review_id": review.get("id") or review.get("reviewId"),
        "company": company,
        "domain": normalize_domain(domain),
        "business_unit_id": business_unit_id,
        "rating": review.get("stars") or review.get("rating"),
        "title": review.get("title"),
        "text": review.get("text") or review.get("comment"),
        "language": review.get("language"),
        "created_at": review.get("createdAt") or review.get("createdDate"),
        "updated_at": review.get("updatedAt") or review.get("updatedDate"),
        "consumer_country": consumer_country,
        "consumer_hash": consumer_hash,
        "source": "trustpilot",
        "ingested_at": datetime.now(timezone.utc).isoformat(),
    }


def fetch_company_reviews(
    company: str,
    domain: str,
    api_key: str,
) -> List[Dict]:
    business_unit_id = find_business_unit_id(domain, api_key)

    rows = []

    for review in iter_public_reviews(
        business_unit_id=business_unit_id,
        api_key=api_key,
        per_page=CONFIG["per_page"],
    ):
        rows.append(
            flatten_review(
                review=review,
                company=company,
                domain=domain,
                business_unit_id=business_unit_id,
            )
        )

    print(f"{company}: total reviews fetched={len(rows)}")

    return rows

In [ ]:
schema = T.StructType([
    T.StructField("review_id", T.StringType()),
    T.StructField("company", T.StringType()),
    T.StructField("domain", T.StringType()),
    T.StructField("business_unit_id", T.StringType()),
    T.StructField("rating", T.IntegerType()),
    T.StructField("title", T.StringType()),
    T.StructField("text", T.StringType()),
    T.StructField("language", T.StringType()),
    T.StructField("created_at", T.StringType()),
    T.StructField("updated_at", T.StringType()),
    T.StructField("consumer_country", T.StringType()),
    T.StructField("consumer_hash", T.StringType()),
    T.StructField("source", T.StringType()),
    T.StructField("ingested_at", T.StringType()),
])

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONFIG['catalog']}.{CONFIG['schema']}")
api_key = get_api_key()
all_rows: List[Dict] = []
errors: List[Dict] = []

for item in COMPANIES:
    company = item["company"]
    domain = item["domain"]

    print(f"\nFetching: {company} ({domain})")

    try:
        rows = fetch_company_reviews(
            company=company,
            domain=domain,
            api_key=api_key,
        )
        all_rows.extend(rows)

    except Exception as exc:
        errors.append({
            "company": company,
            "domain": domain,
            "error": repr(exc),
        })
        print(f"FAILED: {company} ({domain}) -> {repr(exc)}")

if errors:
    print("\nCompanies with errors:")
    for err in errors:
        print(err)

if not all_rows:
    raise RuntimeError("No reviews were fetched. See errors above.")

In [ ]:
raw_df = spark.createDataFrame(all_rows, schema=schema)

clean_df = (
    raw_df
    .withColumn("created_ts", F.to_timestamp("created_at"))
    .withColumn("updated_ts", F.to_timestamp("updated_at"))
    .dropDuplicates(["review_id", "company"])
)

display(clean_df.groupBy("company").count().orderBy("company"))

target_table = f"{CONFIG['catalog']}.{CONFIG['schema']}.{CONFIG['bronze_table']}"

# Full fresh overwrite to clean up any schema pollution from prior runs.
(
    clean_df.write
    .format("delta")
    .mode(CONFIG["write_mode"])
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Wrote {clean_df.count()} reviews to {target_table}")

spark.sql(f"OPTIMIZE {target_table} ZORDER BY (company, created_ts)")

print("Final table counts:")
display(spark.table(target_table).groupBy("company").count().orderBy("company"))

print("Final table schema:")
spark.table(target_table).printSchema()